In [67]:
import os
from dotenv import load_dotenv
from typing import Optional,Literal
from typing_extensions import Dict
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage,HumanMessage,ToolMessage
from langchain_core.prompts import HumanMessagePromptTemplate,SystemMessagePromptTemplate,PromptTemplate,ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser,PydanticOutputParser
from pydantic import BaseModel,Field
from langchain_core.runnables import RunnablePassthrough,RunnableLambda

In [68]:
load_dotenv(override=True)
str_parser = StrOutputParser()
llm = ChatOpenAI(model="gpt-5.4-mini")

##### 4. Parser chain (priority and needs_human)

In [69]:
class TicketInfo(BaseModel):
   priority : Literal["low", "medium", "high"]
   needs_human : bool = Field(description="True if a human agent must handle it")
   summary : str = Field(description="One sentence summary")

In [70]:
parser = PydanticOutputParser(pydantic_object=TicketInfo)

In [71]:
instruction = parser.get_format_instructions()
print(instruction)

The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"priority": {"enum": ["low", "medium", "high"], "title": "Priority", "type": "string"}, "needs_human": {"description": "True if a human agent must handle it", "title": "Needs Human", "type": "boolean"}, "summary": {"description": "One sentence summary", "title": "Summary", "type": "string"}}, "required": ["priority", "needs_human", "summary"]}
```


#### 1. Classifier (router step)

In [72]:
# system_classifier_message = SystemMessage("""
# "Classify the support ticket. "
# "Reply with exactly one word: billing, technical, account or other."
# """)

# user_message = HumanMessage("{ticket_text}")

# system_classifier_prompt = SystemMessagePromptTemplate.from_template(system_classifier_message.content)
# user_message_prompt = HumanMessagePromptTemplate.from_template(user_message.content)

# classifier_template = ChatPromptTemplate.from_messages([system_classifier_prompt,user_message_prompt])

# classifier_chain = classifier_template | llm | parser

classifier_prompt = ChatPromptTemplate.from_messages([
    "system",
     "Classify the support ticket. "
     "Reply with exactly one word: billing, technical, account or other.",
     "human",
     "{ticket_text}"])
classifier_chain = classifier_prompt | llm | str_parser

# tickets = [
#     "I was charged twice this month and nobody replied to my emails!",
#     "App crashes when I upload a photo",
#     "How do I change my email address?",
#     "Do you have a student discount?",
# ]

# for t in tickets:
#     print(classifier_chain.invoke({"ticket_text":t}))

#### 2. Specialist chains

In [73]:
def make_chain(systemPrompt:str):
    return ChatPromptTemplate.from_messages([
     ("system",systemPrompt),
    ("human","{ticket_text}"),
    ]) | llm |str_parser

In [74]:
billing_chain = make_chain(
    "You are a billing agent. Check for double charges, refunds and invoices. "
    "Be polite, apologise if needed, and give clear next steps in 3-4 lines.")

technical_chain = make_chain(
    "You are a tech support engineer. Give numbered troubleshooting steps. "
    "Keep it under 5 steps."
)

account_chain = make_chain(
    "You are an account agent. Help with login, email and password changes. "
    "Give short, clear instructions."
)
general_chain = make_chain(
    "You are a support agent. Answer briefly and politely."
)

#### 3. Route function

In [75]:
def routing(data:dict):
    category = data['category'].strip().lower()
    if "billing" in category:
        return billing_chain
    if "technical" in category:
        return technical_chain
    if "account" in category:
        return account_chain
    return general_chain

reply_router  = RunnableLambda(routing)


In [ ]:
info_chain  = ChatPromptTemplate.from_messages([
       ("system",
     "Analyse the support ticket. "
     "Priority is high for money problems, outages or angry customers."),
     ("human", "{ticket_text}"),
]) | llm.with_structured_output(TicketInfo
                                )

#### 5. Full pipeline

In [77]:
pipeline = (
    RunnablePassthrough.assign(category=classifier_chain) | 
    RunnablePassthrough.assign(reply=reply_router,info=info_chain
                               )
)

##### 6. Test

In [78]:
tickets = [
    "I was charged twice this month and nobody replied to my emails!",
    "App crashes when I upload a photo",
    "How do I change my email address?",
    "Do you have a student discount?",
]

for t in tickets:
    result = pipeline.invoke({"ticket_text": t})
    print("=" * 60)
    print("Ticket    :", t)
    print("Category  :", result["category"])
    print("Priority  :", result["info"].priority)
    print("Human?    :", result["info"].needs_human)
    print("Summary   :", result["info"].summary)
    print("Reply     :", result["reply"])

    if result["info"].priority == "high" or result["info"].needs_human:
        print(">>> Send to human queue")
    else:
        print(">>> Auto-reply")

Ticket    : I was charged twice this month and nobody replied to my emails!
Category  : billing
Priority  : high
Human?    : True
Summary   : Customer reports being charged twice this month and not receiving responses to emails.
Reply     : I’m sorry about the duplicate charge and the lack of reply — that’s frustrating.  
Please send the two charge dates, amounts, and the last 4 digits of the card so I can check for a double charge right away.  
I’ll also look for any related refund or invoice issue and help escalate the missed email thread.  
Once I have those details, I’ll update you on the next steps.
>>> Send to human queue
Ticket    : App crashes when I upload a photo
Category  : technical
Priority  : high
Human?    : True
Summary   : The app crashes when the user uploads a photo, indicating a likely outage or blocking bug that needs human investigation.
Reply     : Try these steps:

1. **Update the app**  
   Make sure you’re on the latest version of the app, then try uploading t